# Data Science for Business - Logit on Micro Mortgages

*Session 4 · ISLP 4.3 · 4.4.2 (Tables 4.4–4.7, Fig. 4.8 ROC) · 5.1.3, 5.1.5*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, RocCurveDisplay, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
import statsmodels.formula.api as smf

In [ ]:
np.random.seed(42)
plt.style.use('fivethirtyeight')

## Problem description

In India, there are about 20 million home loan (mortgage) aspirants
working in the informal sector:

- Monthly income between INR 20,000-25,000 (\$ 325-400)
- Typically no formal accounts and documents (e.g., tax returns, income proofs, bank statements)
- Often use services of money lenders with interest rates between 30 and 60% per annum

Providing mortgages to this group of customers requires to quickly and
efficiently assess their creditworthiness. Due to a lack of formal
documents and objective data, most financial institutions perform
interview-based processes to decide about these loan requests:

Strength of the current process:

-   Interview-based field assessment

-   Relaxation of document requirements

Weaknesses of the current process:

-   Costly (total transaction costs as high as 30% of loan volume)

-   Subjective judgments; depends on individual skills and motivations

-   Low reliability across branches and credit officers

-   Risk of corruption and fraud

The data comes from a Harvard Business Review case. Our goal: predict the credit officers' decision (`Decision`: 1 = approved, 0 = rejected) from the application data. See `variables.pdf` for a description of the variables (source: Rudravaram et al., 2013).

## Load data

Load data from CSV file.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_04/micromortgage.csv')

In [ ]:
data.head()

## Prepare data

We do some minimal data preparation, that is, drop the `ID` column and turn the `Tier` variable into a string, so that it is treated as categorical.

In [ ]:
data = data.drop(['ID'], axis=1)
data["Tier"] = "T" + data["Tier"].astype(str)

All monetary variables (loan amount, down payment, market value of the property, bank savings, and the monthly income, expenses and installments) are in Indian rupees, with values of up to several million. We express them in units of **100,000 rupees** (one *lakh*, a common unit in India). This makes the coefficients of our models much easier to read (e.g., the effect of 100,000 rupees more down payment instead of one rupee more) and avoids numerical problems when fitting models; it does not change the predictions. The mortgage ratios (`IIR`, `IAR`, `FOIR`, `LTV`, `LVR`) are percentages and stay as they are.

In [ ]:
money_columns = ['LoanReq', 'Dwnpay', 'MarVal', 'BankSave', 'TotInc', 'Expen', 'DispInc', 'CalcEmi', 'OldEmi']
data[money_columns] = data[money_columns] / 100000   # in 100,000 rupees

data[money_columns].describe()

Next, we perform a standard 80:20 random train/test split (validation set approach, ISLP 5.1.1). `stratify` keeps the share of approvals equal in both sets.

In [ ]:
train, test = train_test_split(data, test_size=0.2, random_state=42, stratify=data["Decision"])

## Exploratory data analysis

### Descriptive summary statistics

Calculate the base rate of mortgage approvals.

In [ ]:
train["Decision"].mean()

About 80% of applications are approved. That's the benchmark a useful model must beat: always predicting "approve" already gives 80% accuracy.

Note that here class 1 is *approval*, whereas the positive class in ISLP Ch. 4 is *default*. Keep this in mind when we discuss precision and recall below.

### Relationships between response and predictors

Visualize `Decision` against `Age`.

In [ ]:
sns.boxplot(data=train, x="Decision", y="Age")
plt.show()

**Your task:** Plot `TotInc` (total income) or `Dwnpay` (down payment) against `Decision`.

In [ ]:
# YOUR CODE HERE

Visualize `Decision` against `Gender`.

In [ ]:
sns.barplot(data=train, x="Gender", y="Decision")
plt.show()

**Your task:** Compare approval rates by `Tier` or `Employment_Type`.

In [ ]:
# YOUR CODE HERE

## Fit model with statsmodels

Ok, now we are ready to fit our first logistic regression model (ISLP 4.3). Pick one or more of the following predictors: `Age`, `Gender`, `Dwnpay` (down payment), `LoanReq` (requested loan amount) and `Term`. You can include any combination of these predictors (interactions) in your model.

In [ ]:
model_logit = smf.logit(formula='Decision ~ Dwnpay + Age', data=train)
model_logit = model_logit.fit()

In [ ]:
print(model_logit.summary())

## Make predictions

Let's compute predictions, both in the form of probabilities and binary decisions. We classify an application as approved if the predicted probability is at least 0.5. A bank would pick the threshold from the cost of a wrong approval vs. a wrong rejection (ISLP 4.4.2).

In [ ]:
threshold = 0.5
pred_proba = model_logit.predict(test)
pred_label = (pred_proba >= threshold).astype(int)

In [ ]:
pred_proba

In [ ]:
pred_label

## Evaluate accuracy

We begin by calculating some standard metrics like precision, recall, F1-score, and accuracy. Remember that the row for class 1 refers to *approvals*.

In [ ]:
print(classification_report(test["Decision"], pred_label, zero_division=0))

The confusion matrix helps us to diagnose what kind of mistakes the model makes.

In [ ]:
ConfusionMatrixDisplay.from_predictions(test["Decision"], pred_label)
plt.grid(False)
plt.show()

Finally, we plot the ROC curve and compute the AUC, which characterizes the performance of a classifier over all thresholds in one single number.

In [ ]:
RocCurveDisplay.from_predictions(test["Decision"], pred_proba, name="Logistic regression")
plt.show()

## Cross-validation with scikit-learn

### Why cross-validation?

Our test AUC above depends on which rows happened to land in the test set; with another random split, we would get a somewhat different number. $k$-fold cross-validation (ISLP 5.1.3) splits the data into $k$ folds and fits the model $k$ times, each time holding out a different fold for evaluation and training on the remaining $k-1$ folds. Averaging the $k$ results gives a more robust estimate, and their spread shows how much it varies. With $k=5$:

```
Fold 1: [Test ] [Train] [Train] [Train] [Train]
Fold 2: [Train] [Test ] [Train] [Train] [Train]
Fold 3: [Train] [Train] [Test ] [Train] [Train]
Fold 4: [Train] [Train] [Train] [Test ] [Train]
Fold 5: [Train] [Train] [Train] [Train] [Test ]
```

### From formulas to X and y

So far, we have used *statsmodels* with a **formula**: we passed the whole data frame and a formula such as `'Decision ~ Dwnpay + Age'`, and statsmodels picked the right columns itself. From now on, we will mostly use **scikit-learn**, the most popular Python library for machine learning. scikit-learn does not understand formulas. Instead, we hand over the predictors and the response **separately**:

- **`X`** is a table (a *matrix*) with the **predictors only**: one row per loan application and one column per predictor. All columns must be numbers (we will see below how to handle categorical predictors). By convention, it is written with a capital letter because it is a matrix.
- **`y`** is a single column (a *vector*) with the **response**: one value per loan application.

`X` and `y` must have the same rows in the same order: row 1 of `X` and the first value of `y` belong to the same application.

| | statsmodels (formula) | scikit-learn |
|---|---|---|
| Input | data frame + formula | `X` and `y` separately |
| Fit | `smf.logit('Decision ~ ...', data=train).fit()` | `model.fit(X, y)` |
| Predict | `model.predict(test)` | `model.predict_proba(X_test)` |
| Output | summary table with coefficients and p-values | focus on predictions |

For the cross-validation, we use a set of numerical loan characteristics as predictors. Note that this is a different model than above. We only use the training data, so the test set stays untouched.

In [ ]:
X_train_cv = train[['LoanReq', 'Term', 'Dwnpay', 'BankSave', 'CalcEmi', 'IIR', 'IAR', 'FOIR', 'LVR']]
y_train_cv = train['Decision']

In [ ]:
X_train_cv.head()

In [ ]:
y_train_cv.head()

`X_train_cv` has one row per application and one column per predictor; `y_train_cv` has one value per application:

In [ ]:
print('X_train_cv:', X_train_cv.shape)
print('y_train_cv:', y_train_cv.shape)

### Define the model

This time we use scikit-learn's `LogisticRegression` instead of statsmodels. It is the same kind of model as above. The two settings in the code below only make sure that scikit-learn fits the plain logistic regression from the lecture. You don't need to worry about the technical details.

In [ ]:
model_cv = LogisticRegression(C=1e10, solver='newton-cholesky')
model_cv

In scikit-learn, every model works the same way in two steps: first we **create** the model object (above), then we **fit** it to data with `model.fit(X, y)`. Let's do this once by hand on all training data, to see the pattern:

In [ ]:
model_cv.fit(X_train_cv, y_train_cv)

Predictions work with `predict_proba(X)`. It returns **two columns**: the predicted probability of class 0 (rejection) and of class 1 (approval). Each row adds up to 1.

To see what the output looks like, we predict the first five applications of the **training data**. Careful: the model has already seen these applications during fitting, so these are *not* honest predictions for new data, and we should not use them to judge the model. To evaluate a model, we need applications it has not seen, which is exactly what cross-validation provides.

In [ ]:
model_cv.predict_proba(X_train_cv.head())

### Define the folds

`StratifiedKFold` creates folds that all have (about) the same approval rate as the training data (*stratified*). `shuffle=True` assigns the rows to folds at random instead of in their original order; `random_state=42` makes this reproducible.

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

`folds` itself contains no data yet; it is a recipe for splitting. Let's look at the folds it creates for our training data. For each fold, `split` returns the **row numbers** (positions within the training data) used for training and the row numbers held out for evaluation. We print how many rows each part has, the approval rate of the held-out rows and the first few held-out row numbers:

In [ ]:
for fold_number, (training_rows, held_out_rows) in enumerate(folds.split(X_train_cv, y_train_cv), start=1):
    print(f'Fold {fold_number}: {len(training_rows)} rows for training, {len(held_out_rows)} rows held out, '
          f'approval rate held out: {y_train_cv.iloc[held_out_rows].mean():.2f}, '
          f'first held-out rows: {held_out_rows[:5]}')

print(f'Approval rate in the whole training data: {y_train_cv.mean():.2f}')

Every row is held out exactly once, and thanks to stratification, all folds have about the same approval rate. `cross_val_score` below creates exactly these folds internally.

### Run the cross-validation

`cross_val_score` fits the model five times (each time on four folds) and returns one score per held-out fold. Scoring is done using the specified metric, which in this case is the AUC.

Note that we do not call `fit` ourselves here: `cross_val_score` takes care of it and fits a fresh copy of the model on each set of four training folds.

In [ ]:
cv_scores = cross_val_score(model_cv, X_train_cv, y_train_cv, cv=folds, scoring='roc_auc')

print('AUC per fold:', cv_scores.round(4))
print(f'Mean AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f} (standard deviation)')

### Interpret the results

The mean AUC across the five folds is our estimate of how well this model performs on new data; the standard deviation shows how much the AUC varies from fold to fold, i.e., how much a single split could mislead us. Compare the mean with the test AUC of our first model above. Keep in mind that the two models use different predictors: the loan characteristics (e.g., the ratios `FOIR` or `LVR`) seem to carry much more information about the decision than the predictors of our first model.

### Interactions in scikit-learn

In the formula interface, an interaction is easy to request: `'Decision ~ LoanReq * Term'` includes `LoanReq`, `Term` **and** their interaction, i.e., the product `LoanReq × Term` (ISLP 3.3.2). scikit-learn has no formulas, so we create the interaction ourselves: we add a **new column** to `X` that contains the product of the two predictors. For the model, this column is just another predictor.

We take a copy of `X_train_cv`, so that the original table stays unchanged, and add the interaction between the requested loan amount and the term. Since we have expressed all amounts in 100,000 rupees at the beginning, the product stays in a reasonable range (loan amount in 100,000 rupees × term in months).

In [ ]:
X_train_cv_interaction = X_train_cv.copy()
X_train_cv_interaction['LoanReq_x_Term'] = X_train_cv_interaction['LoanReq'] * X_train_cv_interaction['Term']

X_train_cv_interaction.head()

In the same way, we could add further interactions, one column per selected pair of predictors. Now we run exactly the same cross-validation as before (same model, same folds), only with the extended `X`:

In [ ]:
cv_scores_interaction = cross_val_score(model_cv, X_train_cv_interaction, y_train_cv, cv=folds, scoring='roc_auc')

print('AUC per fold:', cv_scores_interaction.round(4))
print(f'Mean AUC with interaction:    {cv_scores_interaction.mean():.4f} ± {cv_scores_interaction.std():.4f}')
print(f'Mean AUC without interaction: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

Because we use the same folds, the two results are directly comparable. Does the interaction improve the mean AUC by more than the fold-to-fold variation (the standard deviation)? If not, the simpler model without the interaction is preferable.

### Categorical predictors in scikit-learn

So far, our `X` contains numerical predictors only. In the formula interface, categorical predictors such as `Gender` were no problem: statsmodels automatically turned them into **dummy variables** (0/1 columns) and used one category as the baseline (ISLP 3.3.1). scikit-learn only accepts numbers, so we have to create the dummy variables ourselves. This is called **one-hot encoding**.

As an example, we add the employer type (`Employer_Type`) and whether the applicant lives in rented accommodation (`Accommodation_Class`). scikit-learn's `OneHotEncoder` works in two steps, just like a model:

- **`fit`** looks at the **training data** and remembers which categories each variable has.
- **`transform`** turns a data set into dummy columns, using exactly these remembered categories.

With `drop='first'`, the encoder leaves out the first category (in alphabetical order), which becomes the baseline, just like in statsmodels. `sparse_output=False` and `set_output(transform='pandas')` make sure we get an ordinary data frame back.

In [ ]:
categorical_columns = ['Employer_Type', 'Accommodation_Class']

encoder = OneHotEncoder(drop='first', sparse_output=False).set_output(transform='pandas')
encoder.fit(train[categorical_columns])

encoder.categories_

These are the categories the encoder has learned from the training data. Now we transform the training data into dummy columns and put them next to the numerical predictors in `X_train_cv`:

In [ ]:
train_dummies = encoder.transform(train[categorical_columns])

X_train_cv_categorical = pd.concat([X_train_cv, train_dummies], axis=1)

X_train_cv_categorical.head()

`Employer_Type` (four categories) has become three dummy columns, with `Business` as the baseline, and `Accommodation_Class` (two categories) has become one column, with `Non_Rented` as the baseline. Now we run the same cross-validation again:

In [ ]:
cv_scores_categorical = cross_val_score(model_cv, X_train_cv_categorical, y_train_cv, cv=folds, scoring='roc_auc')

print('AUC per fold:', cv_scores_categorical.round(4))
print(f'Mean AUC with categorical predictors:    {cv_scores_categorical.mean():.4f} ± {cv_scores_categorical.std():.4f}')
print(f'Mean AUC without categorical predictors: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

Again, compare the difference with the fold-to-fold variation: do the two categorical predictors add anything?

### Apply the same encoding to the test set

To evaluate a model on the test set, the test set must have **exactly the same columns** as the training data, in the same order. That is why we fitted the encoder on the training data: we now only call `transform` on the test set, so the test set is encoded with the categories learned from the training data (we do **not** fit the encoder again). Even if a category happened to be missing in the test set, we would get the same columns.

In [ ]:
test_dummies = encoder.transform(test[categorical_columns])

X_test_categorical = pd.concat([test[X_train_cv.columns], test_dummies], axis=1)
y_test = test['Decision']

print('Same columns as in training:', list(X_test_categorical.columns) == list(X_train_cv_categorical.columns))
X_test_categorical.head()

Finally, we fit the model on the whole training data and compute its AUC on the test set:

In [ ]:
model_categorical = LogisticRegression(C=1e10, solver='newton-cholesky')
model_categorical.fit(X_train_cv_categorical, y_train_cv)

test_probabilities = model_categorical.predict_proba(X_test_categorical)[:, 1]
print(f'Test AUC: {roc_auc_score(y_test, test_probabilities):.4f}')

From Session 5 on, we will bundle such preparation steps and the model into a *pipeline*, which applies `fit` and `transform` in the right places automatically.

## Your turn!

**Your task:** Improve the logistic regression model by including more predictors (e.g., `Tier`, `Employment_Type`, `TotInc`, `LTV`). Did the extra predictors improve the test AUC, or did you overfit?

In [ ]:
# YOUR CODE HERE